In [ ]:
#Imports

from tqdm.auto import tqdm
import numpy as np

from minsearch import VectorSearch, Index

from embedder import Embedder
from gitsource import GithubRepositoryDataReader
from gitsource import chunk_documents


In [ ]:
def rrf(result_lists, k=60, num_results=5):
    '''
    Function for hybrid search using Reciprocal Rank Fusion (RRF) algorithm. It takes a list of result lists from different search methods, combines them, and returns the top results based on their scores.
    '''
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [ ]:
#Q1

embed = Embedder()

q1 = "How does approximate nearest neighbor search work?"

v1 = embed.encode(q1)

v1[0]

np.float64(-0.02058203437252893)

In [ ]:
#Load files from the GitHub repository, filtering for markdown files in the lessons directory.

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

In [19]:
dv = [doc['content'] for doc in documents if doc['filename'] == '02-vector-search/lessons/07-sqlitesearch-vector.md'][0]

In [20]:
v2 = embed.encode(dv)

In [22]:
#Q2
v1.dot(v2)

np.float64(0.36107027225589694)

In [ ]:
chunks_full = chunk_documents(documents, size=2000, step=1000)

chunks = [chunk['content'] for chunk in chunks_full]

In [ ]:
batch_size = 50
X = []

for i in tqdm(range(0, len(chunks), batch_size)):
    batch = chunks[i:i + batch_size]
    batch_vectors = embed.encode_batch(batch)
    X.extend(batch_vectors)

X = np.array(X)

  0%|          | 0/6 [00:00<?, ?it/s]

In [36]:
scores = X.dot(v1)
idx = np.argmax(scores)

top_chunk = chunks[idx]

In [38]:
filename = [chunk['filename'] for chunk in chunks_full if chunk['content'] == top_chunk]

In [ ]:
#Q3
filename

['02-vector-search/lessons/07-sqlitesearch-vector.md']

In [42]:
vindex = VectorSearch(keyword_fields=['filename'])
vindex.fit(X, chunks_full)

In [43]:
query = "What metric do we use to evaluate a search engine?"
query_vector = embed.encode(query)

results = vindex.search(query_vector, num_results=5)

In [46]:
#Q4
results[0]['filename']

'04-evaluation/lessons/05-search-metrics.md'

In [48]:
index  = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

index.fit(chunks_full)

In [50]:
#Q5

query = "How do I store vectors in PostgreSQL?"
query_vector = embed.encode(query)

results_vector = vindex.search(query_vector, num_results=5)
results_kw = index.search(query, num_results=5)


In [ ]:
kw_filenames = {result['filename'] for result in results_kw}
for result in results_vector:
    filename = result['filename']
    if filename not in kw_filenames:
        
        print(f"Vector search result not found in keyword search: {filename}")

Vector search result not found in keyword search: 02-vector-search/lessons/08-pgvector.md
Vector search result not found in keyword search: 02-vector-search/lessons/08-pgvector.md
Vector search result not found in keyword search: 02-vector-search/lessons/08-pgvector.md
Vector search result not found in keyword search: 02-vector-search/lessons/08-pgvector.md


In [61]:
#Q6

query = "How do I give the model access to tools?"
query_vector = embed.encode(query)

results_vector = vindex.search(query_vector, num_results=5)
results_kw = index.search(query, num_results=5)



In [62]:
results = rrf([results_vector, results_kw])

In [63]:
results

[{'start': 4000,
  'content': ' function. `parameters` is a JSON schema\nfor the arguments, and we mark `query` as required so the model always\nfills it in.\n\n## Sending the question with the tool\n\nNow we send the same question as before, but this time we include the\ntool in the request:\n\n```python\nresponse = openai_client.responses.create(\n    model="gpt-5.4-mini",\n    input=messages,\n    tools=[search_tool],\n)\n\nresponse.output\n```\n\nLook at the output. Instead of a message with the answer, the response\ncontains a `function_call` entry. The model decided it needs to search\nthe FAQ before answering. Rather than reply, it asked us to run the\nsearch function first.\n\nLook at the arguments too. The model didn\'t pass our question\nverbatim. It judged the raw question wasn\'t the best query to search\nwith. So it rewrote our enrollment question into search keywords like\n"enroll late join course".\n\n## Executing the function and sending the result back\n\nThe function 